In [1]:
!apt-get update && apt-get install -y colmap

# Tự động tìm và copy mã nguồn
!mkdir -p /kaggle/working/gaussian-splatting
!cp -r $(dirname $(find /kaggle/input/ -name "train_full.py" | head -n 1))/* /kaggle/working/gaussian-splatting/
%cd /kaggle/working/gaussian-splatting

# Tải mã nguồn gốc có cờ --recursive (Lấy luôn thư viện con glm)
!rm -rf submodules/diff-gaussian-rasterization
!git clone --recursive -b dr_aa https://github.com/graphdeco-inria/diff-gaussian-rasterization.git submodules/diff-gaussian-rasterization

!rm -rf submodules/simple-knn
!git clone https://gitlab.inria.fr/bkerbl/simple-knn.git submodules/simple-knn

# Cài đặt
!pip install -q plyfile tqdm
!MAX_JOBS=2 pip install ./submodules/diff-gaussian-rasterization
!MAX_JOBS=2 pip install ./submodules/simple-knn


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,838 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [100 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubunt

In [2]:
%cd /kaggle/working/gaussian-splatting
!python preprocess.py \
    --input "/kaggle/input/datasets/hoangle1236/vai-nvs-data/phase1/public_set" \
    --output "/kaggle/working/cleaned_inputs" \
    --subset HCM0204


/kaggle/working/gaussian-splatting
Undistorting HCM0204 (COLMAP CLI)...
Generating alpha masks (COLMAP CLI on white images)...
/kaggle/working/gaussian-splatting/preprocess.py:156: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba, mode="RGBA").save(out_path)
Đã ghi alpha channel vào 240 ảnh undistort tại /kaggle/working/cleaned_inputs/HCM0204/train/images
  Đã đồng bộ tên 240 ảnh trong images.bin (đổi đuôi sau embed alpha)
scene: HCM0204
Tổng ảnh: 409
Số ảnh missing: 169
Còn lại sau khi xóa: 240
Đã cập nhật reconstruction.


In [3]:
%cd /kaggle/working/gaussian-splatting
!python train_full.py \
    --input_dir "/kaggle/working/cleaned_inputs" \
    --model_dir "/kaggle/working/model_outputs" \
    --iterations 30000 \
    --subset HCM0204


/kaggle/working/gaussian-splatting
Found 5 scenes: ['HCM0181', 'HCM0193', 'HCM0204', 'hcm0031', 'hcm0034']
Filtered to subset (1): ['HCM0204']

=== [1/1] Training scene: HCM0204 ===
Training scene: HCM0204
  source_path = /kaggle/working/cleaned_inputs/HCM0204/train
  model_path  = /kaggle/working/model_outputs/HCM0204
Output folder: /kaggle/working/model_outputs/HCM0204
Reading camera 240/240
Converting point3d.bin to .ply, will happen only the first time you open the scene.
Loading Training Cameras
Loading Test Cameras
Number of points at initialisation :  251715
Training progress:  23%|▏| 7000/30000 [26:55<2:18:18,  2.77it/s, Loss=0.0608529,
[ITER 7000] Evaluating train: L1 0.04198213294148445 PSNR 21.43962745666504

[ITER 7000] Saving Gaussians
Training progress: 100%|█| 30000/30000 [3:16:58<00:00,  2.54it/s, Loss=0.0380512

[ITER 30000] Evaluating train: L1 0.030402906239032745 PSNR 22.590011596679688

[ITER 30000] Saving Gaussians

Done. 1/1 scenes succeeded.


In [4]:
%cd /kaggle/working/gaussian-splatting
!python render_scene.py \
  --orig_dir "/kaggle/input/datasets/hoangle1236/vai-nvs-data/phase1/public_set" \
  --input_dir "/kaggle/working/cleaned_inputs" \
  --model_dir "/kaggle/working/model_outputs" \
  --image_dir "/kaggle/working/image_outputs" \
  --iterations 30000 \
  --scene_name HCM0204


/kaggle/working/gaussian-splatting
Looking for config file in /kaggle/working/model_outputs/HCM0204/cfg_args
Config file found: /kaggle/working/model_outputs/HCM0204/cfg_args
Loading trained model at iteration 30000 [17/07 10:49:46]
[HCM0204] distortion k=0.010010 f=925.51 cx=660.00 cy=494.50 size=(1320x989) [17/07 10:49:57]
[HCM0204] undistorted render canvas f=925.51 cx=660.00 cy=494.50 size=(1320x989) [17/07 10:49:57]
Rendering HCM0204: 100%|████████████████████████| 60/60 [00:09<00:00,  6.03it/s]
Rendered 60 images for HCM0204 from iteration 30000 -> /kaggle/working/image_outputs/HCM0204 [17/07 10:50:07]


In [5]:
%cd /kaggle/working/gaussian-splatting
!python eval_scene.py \
  --input_dir "/kaggle/working/cleaned_inputs" \
  --image_dir "/kaggle/working/image_outputs" \
  --eval_dir "/kaggle/working/eval_outputs" \
  --scene_name HCM0204

!echo "==== ĐIỂM SỐ CỦA BẠN CHO SCENE HCM0204 ===="
!cat /kaggle/working/eval_outputs/HCM0204.json


/kaggle/working/gaussian-splatting
Traceback (most recent call last):
  File "/kaggle/working/gaussian-splatting/eval_scene.py", line 5, in <module>
    import lpips
ModuleNotFoundError: No module named 'lpips'
==== ĐIỂM SỐ CỦA BẠN CHO SCENE HCM0204 ====
cat: /kaggle/working/eval_outputs/HCM0204.json: No such file or directory
